# RAG — Disruptive Architectures

## Objetivo

Desenvolver um sistema de Retrieval-Augmented Generation (RAG)
capaz de responder perguntas sobre os conteúdos da disciplina
Disruptive Architectures utilizando como base de conhecimento
o site oficial da disciplina.

O sistema realiza a coleta e processamento das páginas,
divide os conteúdos em trechos, transforma esses trechos em
embeddings e realiza recuperação semântica para fornecer
contexto a um modelo de linguagem.

As respostas são geradas com base nas informações recuperadas
da base de conhecimento e acompanhadas de suas respectivas fontes.

In [1]:
!pip install -q google-genai beautifulsoup4 requests python-dotenv

In [2]:
import os
import requests
import numpy as np

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag, urlparse

from google import genai
from google.genai import types

# A chave nunca fica escrita no notebook:
# no Colab vem dos Secrets; localmente, do arquivo .env (fora do git).
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise RuntimeError("Defina a variável GEMINI_API_KEY (Colab Secrets ou arquivo .env).")

client = genai.Client(api_key=API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"

print("Cliente criado com sucesso.")

Cliente criado com sucesso.


## 1. Base de conhecimento

A base de conhecimento utilizada pelo RAG é o site oficial
da disciplina Disruptive Architectures.

O sistema identifica as páginas internas do site e utiliza
seus conteúdos textuais como fonte para a recuperação semântica.

A coleta **não é um crawler recursivo**: os links são extraídos
apenas da página inicial. Isso é suficiente porque o menu de
navegação do site, presente na página inicial, já lista as páginas
da disciplina. Páginas que não aparecem nesse menu não entram na base.

In [3]:
BASE_URL = "https://arnaldojr.github.io/DisruptiveArchitectures/"

response = requests.get(BASE_URL, timeout=10)
response.raise_for_status()

print(f"Site acessado com sucesso: {response.status_code}")

Site acessado com sucesso: 200


In [4]:
def coletar_documento(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    texto = soup.get_text("\n", strip=True)

    return {
        "url": url,
        "texto": texto
    }

In [5]:
soup = BeautifulSoup(response.text, "html.parser")

links_internos = []

for a in soup.find_all("a", href=True):
    # Remove âncoras (#secao) para não coletar a mesma página duas vezes.
    url, _ = urldefrag(urljoin(BASE_URL, a["href"]))
    parsed = urlparse(url)

    if (
        parsed.netloc == "arnaldojr.github.io"
        and parsed.path.startswith("/DisruptiveArchitectures/")
    ):
        links_internos.append(url)

links_internos = sorted(set(links_internos))

print(f"Páginas internas encontradas: {len(links_internos)}")

Páginas internas encontradas: 28


In [6]:
DOCUMENTOS = []

for url in links_internos:
    try:
        documento = coletar_documento(url)

        if documento["texto"]:
            DOCUMENTOS.append(documento)

    except Exception as e:
        print(f"Erro ao acessar {url}: {e}")

print(f"Documentos coletados: {len(DOCUMENTOS)}")

Documentos coletados: 28


## 2. Divisão dos documentos em chunks

Os documentos coletados são divididos em trechos menores
(chunks) para facilitar a recuperação das informações
relevantes.

Utilizamos sobreposição (overlap) entre os chunks para
preservar parte do contexto entre trechos consecutivos.

In [7]:
def criar_chunks(texto, tamanho=1000, overlap=200):
    if overlap >= tamanho:
        raise ValueError("overlap precisa ser menor que tamanho")

    chunks = []

    inicio = 0

    while inicio < len(texto):
        fim = inicio + tamanho
        chunk = texto[inicio:fim]

        chunks.append(chunk)

        # Sem isso, o último chunk seria só um pedaço do overlap,
        # já contido inteiro no chunk anterior.
        if fim >= len(texto):
            break

        inicio = fim - overlap

    return chunks

In [8]:
CHUNKS = []

for documento in DOCUMENTOS:
    chunks = criar_chunks(documento["texto"])

    for i, chunk in enumerate(chunks):
        CHUNKS.append({
            "url": documento["url"],
            "chunk_id": i,
            "texto": chunk
        })

print(f"Total de chunks: {len(CHUNKS)}")

Total de chunks: 207


## 3. Geração dos embeddings

Cada chunk é convertido em um embedding, ou seja, uma
representação numérica do seu significado semântico.

Esses vetores serão utilizados posteriormente para comparar
a pergunta do usuário com os conteúdos da base de conhecimento.

O tipo de tarefa é informado pelo parâmetro `task_type` da API
de embeddings: `RETRIEVAL_DOCUMENT` para os chunks armazenados no
índice e, mais adiante, `RETRIEVAL_QUERY` para a pergunta do usuário.
Assim o modelo gera vetores otimizados para busca documento ↔ consulta.


In [9]:

# Os embeddings são gerados em lotes para reduzir a quantidade de
# requisições à API e tornar o processamento mais eficiente.
import time

def gerar_embeddings(chunks, tamanho_lote=50, max_tentativas=3):
    indice = []

    for inicio in range(0, len(chunks), tamanho_lote):
        lote = chunks[inicio:inicio + tamanho_lote]

        textos = [chunk["texto"] for chunk in lote]

        for tentativa in range(max_tentativas):
            try:
                resultado = client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=textos,
                    config=types.EmbedContentConfig(
                        task_type="RETRIEVAL_DOCUMENT"
                    )
                )

                break

            except Exception as e:
                if "429" not in str(e):
                    raise

                if tentativa == max_tentativas - 1:
                    raise

                print(
                    f"Quota atingida. "
                    f"Aguardando 60 segundos antes de tentar novamente..."
                )

                time.sleep(60)

        for chunk, embedding in zip(lote, resultado.embeddings):
            indice.append({
                "url": chunk["url"],
                "chunk_id": chunk["chunk_id"],
                "texto": chunk["texto"],
                "embedding": np.array(
                    embedding.values,
                    dtype=float
                )
            })

        processados = min(inicio + tamanho_lote, len(chunks))
        print(f"Processados: {processados}/{len(chunks)}")

    return indice

In [10]:
INDICE = gerar_embeddings(CHUNKS)

print(f"\nÍndice criado: {len(INDICE)} chunks")
print(f"Dimensão dos embeddings: {len(INDICE[0]['embedding'])}")

Processados: 50/207
Processados: 100/207
Quota atingida. Aguardando 60 segundos antes de tentar novamente...
Processados: 150/207
Quota atingida. Aguardando 60 segundos antes de tentar novamente...
Processados: 200/207
Processados: 207/207

Índice criado: 207 chunks
Dimensão dos embeddings: 3072


## 4. Recuperação semântica

Para realizar uma busca, a pergunta do usuário também é
transformada em um embedding.

O vetor da pergunta é comparado com os vetores armazenados
no índice utilizando similaridade de cosseno.

Os chunks com maior similaridade são selecionados como
contexto para a geração da resposta.

In [11]:
def similaridade_cosseno(
    vetor_a: np.ndarray,
    vetor_b: np.ndarray
) -> float:

    denominador = np.linalg.norm(vetor_a) * np.linalg.norm(vetor_b)

    if denominador == 0:
        return 0.0

    return float(np.dot(vetor_a, vetor_b) / denominador)

In [12]:
def recuperar_chunks(pergunta: str, top_k: int = 5):

    resultado = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=pergunta,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

    vetor_pergunta = np.array(
        resultado.embeddings[0].values,
        dtype=float
    )

    resultados = []

    for item in INDICE:

        score = similaridade_cosseno(
            vetor_pergunta,
            item["embedding"]
        )

        resultados.append({
            "url": item["url"],
            "chunk_id": item["chunk_id"],
            "texto": item["texto"],
            "score": score
        })

    resultados.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return resultados[:top_k]

In [13]:
def mostrar_resultados_recuperados(resultados):

    for i, resultado in enumerate(resultados, start=1):

        print(f"\nResultado {i}")
        print(f"Score: {resultado['score']:.4f}")
        print(f"Chunk: {resultado['chunk_id']}")
        print(f"Fonte: {resultado['url']}")
        print(f"Trecho: {resultado['texto'][:500]}...")

In [14]:
resultados_mqtt = recuperar_chunks(
    "O que é MQTT?",
    top_k=5
)

mostrar_resultados_recuperados(resultados_mqtt)


Resultado 1
Score: 0.8261
Chunk: 2
Fonte: https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab23/
Trecho: mas usando MQTT.
O que é MQTT
MQTT é um protocolo de mensagens muito utilizado em IoT. Ao contrário do modelo HTTP, em que um cliente faz uma requisição para um servidor, no MQTT a comunicação acontece por meio de um
broker
e de
tópicos
.
A lógica básica é esta:
um dispositivo pode
publicar
uma mensagem em um tópico;
outro dispositivo pode
assinar
esse tópico;
o broker recebe a mensagem e a entrega para quem estiver inscrito.
Em uma visão prática, pense assim:
o
broker
é o intermediário;
o
tópic...

Resultado 2
Score: 0.7819
Chunk: 1
Fonte: https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab23/
Trecho:  Estruturadas
Lab3.5 - Do protótipo ao produto
Lab4 - RAG e Bases de Conhecimento
Home
1º Semestre - IoT
ESP32
Lab09 - MQTT
MQTT com ESP32
Visão geral da aula
Já sabemos fazer o ESP32 atuar como
servidor e clienteweb
, servindo páginas, expondo e requisita

## 5. Geração da resposta

Os chunks recuperados são utilizados como contexto para o
modelo de linguagem.

O modelo recebe a pergunta e os trechos recuperados e deve
responder utilizando somente as informações encontradas
na base de conhecimento.

Quando a informação não estiver presente no contexto,
o sistema deve informar que ela não foi encontrada na base.

In [25]:
def gerar_resposta(pergunta, resultados):

    contexto = "\n\n".join(
        [
            f"Fonte: {resultado['url']}\n{resultado['texto']}"
            for resultado in resultados
        ]
    )

    prompt = f"""
Você é um assistente da disciplina Disruptive Architectures.

Responda à pergunta utilizando SOMENTE as informações
presentes no contexto abaixo.

Não utilize conhecimento externo ao contexto.
Não invente informações que não estejam presentes no contexto.

Se a informação necessária para responder não estiver
presente no contexto, informe claramente que ela não foi
encontrada na base de conhecimento.

Se a pergunta for ambígua, estiver incompleta ou não for
possível identificar com segurança a que informação o usuário
se refere, peça informações adicionais ao usuário.

Nunca forneça uma informação que não possa ser fundamentada
pelo contexto recuperado.

Seja objetivo e responda em português.

Pergunta:
{pergunta}

Contexto:
{contexto}
"""

    resposta = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    return resposta.text

In [26]:
resposta = gerar_resposta(
    "O que é MQTT?",
    resultados_mqtt
)

print("\nRESPOSTA:")
print(resposta)


RESPOSTA:
O MQTT é um protocolo de mensagens muito utilizado em IoT. Ao contrário do modelo HTTP (em que um cliente faz uma requisição para um servidor), no MQTT a comunicação é orientada a publicação e assinatura, acontecendo por meio de um broker e de tópicos. Nele, os clientes se conectam ao broker, um dispositivo pode publicar uma mensagem em um tópico, outro dispositivo pode assinar (ficar ouvindo) esse tópico, e o broker recebe a mensagem e a entrega para quem estiver inscrito.


## 6. Pipeline completo

A função abaixo integra as etapas de recuperação e geração.

O fluxo realizado pelo sistema é:

Pergunta → Embedding da pergunta → Recuperação dos chunks →
Contexto → Geração da resposta → Fontes

In [29]:
def responder(pergunta, top_k=5):

    resultados = recuperar_chunks(
        pergunta,
        top_k=top_k
    )

    resposta = gerar_resposta(
        pergunta,
        resultados
    )

    print("RESPOSTA")
    print(resposta)

    print("\nFONTES")

    fontes = []

    for resultado in resultados:

        if resultado["url"] not in fontes:
            fontes.append(resultado["url"])

    for fonte in fontes:
        print(f"- {fonte}")

    return resposta

In [30]:
responder("O que é MQTT?")

RESPOSTA
MQTT é um protocolo de mensagens muito utilizado em IoT. Ao contrário do modelo HTTP (em que um cliente faz uma requisição para um servidor), no MQTT a comunicação é orientada a publicação e assinatura, acontecendo por meio de um broker e de tópicos. 

Nele, os clientes se conectam a um broker, as mensagens circulam por tópicos, um dispositivo pode publicar uma mensagem em um tópico e outro dispositivo pode assinar esse tópico para receber a entrega feita pelo broker.

FONTES
- https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab23/
- https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab6/


'MQTT é um protocolo de mensagens muito utilizado em IoT. Ao contrário do modelo HTTP (em que um cliente faz uma requisição para um servidor), no MQTT a comunicação é orientada a publicação e assinatura, acontecendo por meio de um broker e de tópicos. \n\nNele, os clientes se conectam a um broker, as mensagens circulam por tópicos, um dispositivo pode publicar uma mensagem em um tópico e outro dispositivo pode assinar esse tópico para receber a entrega feita pelo broker.'

## 7. Testes qualitativos

Os testes abaixo são uma **demonstração manual**: fazemos quatro
perguntas e comparamos a resposta obtida com o comportamento
esperado. Não são uma avaliação quantitativa do RAG (não há métricas
de recuperação nem verificação automática das respostas).

Cada teste observa um comportamento diferente:

- Informação direta: verifica se o sistema recupera uma
  informação presente na base.
- Paráfrase: verifica se a busca semântica encontra a
  informação mesmo quando a pergunta é formulada de outra forma.
- Informação ausente: verifica se o sistema reconhece quando
  a informação não está disponível na base.
- Ambiguidade: verifica se o sistema evita inventar uma resposta
  quando a pergunta não possui contexto suficiente.

In [31]:
testes = [
    {
        "tipo": "Informação direta",
        "pergunta": "O que é MQTT?",
        "esperado": "Explicar MQTT citando a aula de MQTT como fonte.",
    },
    {
        "tipo": "Paráfrase",
        "pergunta": "Como os dispositivos podem trocar mensagens usando MQTT?",
        "esperado": "Encontrar o mesmo conteúdo do teste 1, mesmo com outras palavras.",
    },
    {
        "tipo": "Informação ausente",
        "pergunta": "Qual é o procedimento para configurar uma rede 5G privada?",
        "esperado": "Dizer que a informação não foi encontrada na base.",
    },
    {
        "tipo": "Ambiguidade",
        "pergunta": "Qual é o prazo?",
        "esperado": "Não escolher um prazo; pedir que o usuário especifique a atividade.",
    },
]

for numero, teste in enumerate(testes, start=1):

    print(f"\n--- TESTE {numero}: {teste['tipo']} ---")
    print("Você:", teste["pergunta"])
    print("Esperado:", teste["esperado"])
    print()

    responder(teste["pergunta"])


--- TESTE 1 ---
Você: O que é MQTT?
RESPOSTA
O MQTT é um protocolo de mensagens muito utilizado em IoT (Internet das Coisas). 

Ao contrário do modelo HTTP (em que um cliente faz uma requisição para um servidor e a comunicação é orientada a requisição/resposta), no MQTT a comunicação é orientada a **publicação** e **assinatura**, acontecendo por meio de um **broker** e de **tópicos**. 

Na lógica do MQTT:
* Um dispositivo pode **publicar** (enviar) uma mensagem em um tópico (canal de comunicação);
* Outro dispositivo pode **assinar** (ficar ouvindo) esse tópico;
* O **broker** atua como o elemento central/intermediário, recebendo a mensagem e a entregando para quem estiver inscrito (clientes inscritos nos tópicos correspondentes).

FONTES
- https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab23/
- https://arnaldojr.github.io/DisruptiveArchitectures/aulas/iot/lab6/

--- TESTE 2 ---
Você: Como os dispositivos podem trocar mensagens usando MQTT?
RESPOSTA
Utilizando o protoco

## 8. Resultados observados

Na execução acima, o sistema:

- respondeu à pergunta direta usando a aula de MQTT como fonte;
- encontrou o mesmo conteúdo quando a pergunta foi reformulada;
- informou que não encontrou na base o procedimento de rede 5G privada;
- diante de "Qual é o prazo?", **não escolheu um prazo**: recuperou
  páginas relacionadas (agenda e checkpoints), mas pediu que o usuário
  especificasse a atividade, como orientado no prompt.

Como as respostas vêm de um modelo generativo, o texto exato pode
variar entre execuções. Estes resultados mostram o comportamento
nestes quatro casos, não uma garantia para qualquer pergunta.



## 9. Conclusão

O projeto implementou um sistema de Retrieval-Augmented Generation
(RAG) utilizando como base de conhecimento o site oficial da
disciplina Disruptive Architectures.

O sistema realiza a coleta das páginas, divisão dos conteúdos em
chunks, geração de embeddings, armazenamento em um índice vetorial,
recuperação semântica por similaridade de cosseno e geração de
respostas utilizando os trechos recuperados como contexto.

Nos testes qualitativos, o sistema recuperou informações relevantes,
lidou com uma pergunta reformulada, reconheceu uma informação ausente
e pediu esclarecimento diante de uma pergunta ambígua.

Limitações conhecidas:

- a coleta usa apenas os links da página inicial (não é recursiva);
- o texto extraído inclui o menu de navegação de cada página, o que
  adiciona ruído aos chunks;
- o índice fica em memória e precisa ser recriado a cada execução;
- a avaliação é manual, sem métricas de recuperação.